# Demo: Bayesian Optimisation for Directed Cell Migration

Complete end-to-end demo using the **virtual microscope migration
backend**. Simulated cells actually migrate in response to
stimulation -- the full rtm-pymmcore pipeline runs for real
(segmentation, tracking, feature extraction, stimulation).

### To run on a real microscope

Replace the microscope setup cell with your real microscope class
and adjust the channel names. Everything else stays the same.

---

**Steerable parameters (BO optimises):**
- `stim_y` -- vertical position of stimulation dot (0=bottom, 1=top of cell)
- `stim_x` -- horizontal position (0=left, 1=right of cell)

**Covariate (observed per cell):**
- `ref_mean_intensity` -- expression level measured via DAPI/ref channel.
  Each simulated cell has a random expression level; higher expression
  = stronger migration response.

**Objective:** maximise `displacement` (px)

### How the simulation works

The `MigrationCell` (in `virtual-microscope`) extends `OptogeneticCell`
with:
- **Asymmetric response**: impulse is scaled by a Gaussian peaked at
  `stim_y_rel = 0.8` (top of cell). Bottom stimulation → no migration.
- **Expression scaling**: per-cell random expression (0.2--1.0) multiplies
  the impulse. Visible as DAPI channel intensity.
- Displacement is measured from **real tracking data** (not synthetic).

In [ ]:
import os
import logging
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Suppress JAX/XLA debug messages
os.environ["JAX_LOG_COMPILES"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import jax

jax.config.update("jax_log_compiles", False)
for _name in list(logging.Logger.manager.loggerDict):
    if "jax" in _name or "absl" in _name:
        logging.getLogger(_name).setLevel(logging.WARNING)

from rtm_pymmcore.core.data_structures import (
    Channel,
    RTMSequence,
    SegmentationMethod,
)
from rtm_pymmcore.core.controller import Controller
from rtm_pymmcore.core.pipeline import ImageProcessingPipeline
from rtm_pymmcore.core.writers import TiffWriter
from rtm_pymmcore.agents.bo_optimization import (
    BOptGPAX,
    BO_Parameter,
    BO_Objective,
    BO_Covariate,
)

## 1. Microscope setup

Uses the `virtual-microscope` **migration** backend. Cells have
per-cell expression and asymmetric migration response.

Replace this cell with your real microscope for actual experiments.

In [ ]:
# ---- Simulated microscope with migration-response cells ----
from virtual_microscope.backends.migration import setup_migration
from rtm_pymmcore.microscope.simulation import UniMMCoreSimulation
import rtm_pymmcore.core.utils as utils

core, sim = setup_migration(n_cells=30, seed=42)
mic = UniMMCoreSimulation(mmc=core)
mic.init_scope()
utils.print_configs(core)

# Show per-cell expression distribution
from virtual_microscope.sims.cell.migration import MigrationCell

exprs = [c.expression for c in sim._cells if isinstance(c, MigrationCell)]
print(
    f"\n{len(exprs)} MigrationCells, expression range: [{min(exprs):.2f}, {max(exprs):.2f}]"
)

# ---- For a real microscope, replace the above with: ----
# from rtm_pymmcore.microscope.pymmcore import PyMMCoreMicroscope
# mic = PyMMCoreMicroscope(config_file="path/to/config.cfg")

## 2. Pipeline setup

In [ ]:
from rtm_pymmcore.segmentation.base import OtsuSegmentator
from rtm_pymmcore.feature_extraction.simple import SimpleFE
from rtm_pymmcore.feature_extraction.ref import RefFE
from rtm_pymmcore.tracking.trackpy import TrackerTrackpy
from rtm_pymmcore.stimulation.dot_on_cell import StimDotOnCell

# --- Experiment timing ---
N_FRAMES = 15  # 3 baseline + 8 stim + 4 post-stim
FIRST_FRAME_STIM = 3
LAST_FRAME_STIM = 11
TIME_BETWEEN_TIMESTEPS = 2  # seconds (fast for demo)

# --- Channels ---
# Virtual microscope: "phase-contrast", "DAPI", "membrane"
# For real mic: replace with your channel config names
IMG_CHANNEL = Channel(config="phase-contrast", exposure=50)
STIM_CHANNEL = Channel(config="phase-contrast", exposure=100)
REF_CHANNEL = Channel(config="DAPI", exposure=100)  # shows per-cell expression

# --- Storage ---
path = os.path.join(tempfile.gettempdir(), "rtm_demo_bo_migration")
os.makedirs(path, exist_ok=True)

# --- Pipeline ---
pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=[
        SegmentationMethod(
            "labels",
            OtsuSegmentator(),
            use_channel=0,
            save_tracked=True,
        )
    ],
    feature_extractor=SimpleFE("labels"),
    tracker=TrackerTrackpy(search_range=15),
    stimulator=StimDotOnCell(radius=8),  # places dot at (stim_y, stim_x) on each cell
    feature_extractor_ref=RefFE("labels"),  # extracts ref_mean_intensity per cell
)

# --- FOV positions ---
stage_positions = [(0.0, 0.0, 0.0)]
fov_indices = list(range(len(stage_positions)))

print(f"Pipeline output: {path}")
print(
    f"Cycle: {FIRST_FRAME_STIM} baseline + {LAST_FRAME_STIM - FIRST_FRAME_STIM} stim"
    f" + {N_FRAMES - LAST_FRAME_STIM} post-stim = {N_FRAMES} frames"
)

## 3. BO agent subclass

Implements the two abstract methods:
- `_create_events_for_cycle` -- builds `RTMEvent` objects with stim parameters in metadata
- `_preprocess_results` -- reads **real** tracking data and computes actual displacement

This code is identical for simulated and real microscopes.

In [ ]:
class MigrationBO(BOptGPAX):
    """BO agent for directed cell migration.

    Identical code for simulated and real microscopes.
    """

    def __init__(
        self,
        *,
        n_frames,
        first_frame_stim,
        last_frame_stim,
        time_between_timesteps,
        img_channel,
        stim_channel,
        ref_channel,
        stage_positions,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.n_frames = n_frames
        self.first_frame_stim = first_frame_stim
        self.last_frame_stim = last_frame_stim
        self.time_between_timesteps = time_between_timesteps
        self.img_channel = img_channel
        self.stim_channel = stim_channel
        self.ref_channel = ref_channel
        self.stage_positions = stage_positions
        self._phase_counter = 0

    def _create_events_for_cycle(self, parameters: dict) -> list:
        """Build RTMEvents for one BO iteration."""
        phase_id = self._phase_counter

        acq = RTMSequence(
            time_plan={
                "interval": self.time_between_timesteps,
                "loops": self.n_frames,
            },
            stage_positions=self.stage_positions,
            channels=(self.img_channel,),
            stim_channels=(self.stim_channel,),
            stim_frames=range(self.first_frame_stim, self.last_frame_stim),
            ref_channels=(self.ref_channel,),
            ref_frames={self.n_frames - 1},  # optocheck at last frame
            rtm_metadata={
                "phase_name": f"BO_iter_{phase_id}",
                "phase_id": phase_id,
                # BO parameters -- read by StimDotOnCell and stored in tracks
                "stim_y": parameters["stim_y"],
                "stim_x": parameters["stim_x"],
            },
        )
        return list(acq)

    def _preprocess_results(self, fov_tracks: dict) -> pd.DataFrame:
        """Compute real displacement from tracking data."""
        phase_id = self._phase_counter
        self._phase_counter += 1

        results = []
        for fov_idx, df_tracks in fov_tracks.items():
            if df_tracks.empty or "particle" not in df_tracks.columns:
                continue

            stim_y = (
                df_tracks["stim_y"].iloc[0] if "stim_y" in df_tracks.columns else 0.5
            )
            stim_x = (
                df_tracks["stim_x"].iloc[0] if "stim_x" in df_tracks.columns else 0.5
            )

            for particle_id, grp in df_tracks.groupby("particle"):
                grp = grp.sort_values("timestep")

                if grp["timestep"].nunique() < 4:
                    continue

                baseline = grp[grp["timestep"] < self.first_frame_stim]
                end = grp[grp["timestep"] >= self.n_frames - 3]

                if len(baseline) < 2 or len(end) < 2:
                    continue

                baseline_area = baseline["area"].median()
                if pd.isna(baseline_area) or baseline_area < 10:
                    continue

                # Real displacement from tracking
                displacement = float(
                    np.sqrt(
                        (end["x"].median() - baseline["x"].median()) ** 2
                        + (end["y"].median() - baseline["y"].median()) ** 2
                    )
                )

                # Covariate: expression from ref channel
                ref_intensity = 0.0
                if "ref_mean_intensity" in grp.columns:
                    vals = grp["ref_mean_intensity"].dropna()
                    if not vals.empty:
                        ref_intensity = float(vals.median())

                results.append(
                    {
                        "stim_y": stim_y,
                        "stim_x": stim_x,
                        "ref_mean_intensity": ref_intensity,
                        "displacement": displacement,
                    }
                )

        if not results:
            print(f"  Phase {phase_id}: no valid cells")
            return pd.DataFrame()

        df = pd.DataFrame(results)
        print(
            f"  Phase {phase_id}: {len(df)} cells, "
            f"mean displacement={df['displacement'].mean():.1f} px"
        )
        return df

## 4. Configure and run BO

In [ ]:
bo_params = [
    BO_Parameter(name="stim_y", bounds=(0.0, 1.0), spacing=0.1),
    BO_Parameter(name="stim_x", bounds=(0.0, 1.0), spacing=0.1),
]
bo_covariates = [BO_Covariate(name="ref_mean_intensity")]
bo_objective = BO_Objective(name="displacement", goal="maximize")

agent = MigrationBO(
    # Pipeline / storage
    storage_path=path,
    # Experiment parameters
    n_frames=N_FRAMES,
    first_frame_stim=FIRST_FRAME_STIM,
    last_frame_stim=LAST_FRAME_STIM,
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,
    img_channel=IMG_CHANNEL,
    stim_channel=STIM_CHANNEL,
    ref_channel=REF_CHANNEL,
    stage_positions=stage_positions,
    # BO hyperparameters
    parameters_to_optimize=bo_params,
    objective_metric=bo_objective,
    bo_covariates=bo_covariates,
    n_iterations=10,
    acquisition_function="ei",
    n_cov_samples=20,
    ei_xi=0.2,
    ei_xi_final=0.01,
    ei_xi_decay_fraction=0.7,
    verbose=True,
)
agent.add_fovs(fov_indices)

# --- Controller ---
writer = TiffWriter(storage_path=path)
ctrl = Controller(mic, pipeline, writer=writer, agent=agent)

# --- Run the BO loop ---
agent.run()

## 5. Inspect generated images

Real images from the virtual microscope, with real segmentation and stim masks.

In [ ]:
import tifffile
from glob import glob

raw_files = sorted(glob(os.path.join(path, "raw", "*.tiff")))
label_files = sorted(glob(os.path.join(path, "labels", "*.tiff")))
stim_files = sorted(glob(os.path.join(path, "stim_mask", "*.tiff")))

n_show = min(4, len(raw_files))
if n_show > 0:
    fig, axes = plt.subplots(3, n_show, figsize=(4 * n_show, 10))
    if n_show == 1:
        axes = axes.reshape(-1, 1)

    for i in range(n_show):
        img = tifffile.imread(raw_files[i])
        if img.ndim == 3:
            img = img[0]
        axes[0, i].imshow(img, cmap="gray")
        axes[0, i].set_title(os.path.basename(raw_files[i]), fontsize=8)
        axes[0, i].axis("off")

        if i < len(label_files):
            axes[1, i].imshow(tifffile.imread(label_files[i]), cmap="nipy_spectral")
            axes[1, i].set_title("Segmentation", fontsize=8)
            axes[1, i].axis("off")

        if i < len(stim_files):
            axes[2, i].imshow(tifffile.imread(stim_files[i]), cmap="hot")
            axes[2, i].set_title("Stim mask", fontsize=8)
            axes[2, i].axis("off")

    plt.suptitle("Pipeline output: raw / segmentation / stim masks")
    plt.tight_layout()
    plt.show()
else:
    print("No images found.")

In [ ]:
# Inspect tracking data
track_file = os.path.join(path, "tracks", "0_latest.parquet")
if os.path.exists(track_file):
    df = pd.read_parquet(track_file)
    print(
        f"Tracked {df['particle'].nunique()} cells across {df['timestep'].nunique()} frames"
    )
    print(f"Columns: {list(df.columns)}")
    if "ref_mean_intensity" in df.columns:
        ref_vals = df.groupby("particle")["ref_mean_intensity"].first().dropna()
        print(
            f"ref_mean_intensity per cell: [{ref_vals.min():.1f}, {ref_vals.max():.1f}]"
        )
    display(df.head(10))
else:
    print("No tracking data yet.")

## 6. BO results

In [ ]:
if agent.x is not None and agent.y is not None:
    x_data = np.array(agent.x)
    y_data = np.array(agent.y).ravel()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # All measurements
    sc = axes[0].scatter(
        x_data[:, 0],
        x_data[:, 1],
        c=y_data,
        cmap="viridis",
        s=20,
        edgecolors="k",
        linewidths=0.3,
    )
    axes[0].set_xlabel("stim_y (0=bottom, 1=top)")
    axes[0].set_ylabel("stim_x")
    axes[0].set_title("All measurements (real displacement)")
    fig.colorbar(sc, ax=axes[0], label="displacement (px)")

    # Convergence
    if agent._iteration_means:
        axes[1].plot(agent._iteration_means, "o-")
        axes[1].set_xlabel("Iteration")
        axes[1].set_ylabel("Mean displacement (px)")
        axes[1].set_title("BO convergence")

    plt.tight_layout()
    plt.show()
else:
    print("No data collected.")

In [ ]:
# 3D GP-predicted landscape
import gpax

rng_key, rng_key_pred = gpax.utils.get_keys()

sy = np.linspace(0, 1, 20)
sx = np.linspace(0, 1, 20)
expr_obs = agent.df_results["ref_mean_intensity"].values
expr = np.linspace(max(expr_obs.min(), 0.01), expr_obs.max(), 10)

grid = np.array([[y, x, e] for y in sy for x in sx for e in expr])
grid_scaled = agent._x_scaler.transform(grid)
y_pred_scaled, _ = agent.model.predict(rng_key_pred, grid_scaled, noiseless=True)
y_pred = agent._y_scaler.inverse_transform(y_pred_scaled).flatten()

fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection="3d")
sc = ax.scatter(
    grid[:, 0],
    grid[:, 1],
    grid[:, 2],
    c=np.asarray(y_pred),
    cmap="viridis",
    s=8,
    alpha=0.5,
)
ax.set_xlabel("stim_y (0=bottom, 1=top)")
ax.set_ylabel("stim_x")
ax.set_zlabel("ref_mean_intensity (expression)")
ax.set_title("GP-predicted displacement landscape")
fig.colorbar(sc, ax=ax, shrink=0.6, pad=0.12, label="displacement (px)")

opt_idx = int(np.argmax(np.asarray(y_pred)))
ax.scatter(
    grid[opt_idx, 0],
    grid[opt_idx, 1],
    grid[opt_idx, 2],
    c="red",
    s=300,
    marker="*",
    edgecolors="black",
    linewidths=2,
    label=(
        f"GP optimum: stim_y={grid[opt_idx,0]:.2f}, " f"stim_x={grid[opt_idx,1]:.2f}"
    ),
    zorder=5,
)
ax.legend()
plt.tight_layout()
plt.show()

print(f"GP-predicted optimum:")
print(f"  stim_y = {grid[opt_idx, 0]:.2f}  (expected: ~0.80)")
print(f"  stim_x = {grid[opt_idx, 1]:.2f}  (expected: ~0.50)")
print(f"  predicted displacement = {float(y_pred[opt_idx]):.1f} px")